# 01 - Exploratory Data Analysis (EDA)
Project N22 - Stroke Prediction. Uses the reusable helpers in `src/eda.py`.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fossbk-spec/stroke-prediction/blob/main/notebooks/01_exploratory_data_analysis.ipynb)
[![Open In Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/fossbk-spec/stroke-prediction/blob/main/notebooks/01_exploratory_data_analysis.ipynb)


In [1]:
# Runs offline on Google Colab / Kaggle Notebooks: no internet/Kaggle token needed.
# The dataset CSV is already bundled in the repo at data/raw/.
import os, sys, subprocess
if "google.colab" in sys.modules or os.path.exists("/kaggle/working"):
    if not os.path.exists("stroke-prediction"):
        subprocess.run(["git", "clone", "--depth", "1",
                         "https://github.com/fossbk-spec/stroke-prediction.git"], check=True)
    os.chdir("stroke-prediction")
    subprocess.run(["pip", "install", "-q", "-r", "requirements.txt"], check=False)
sys.path.insert(0, os.getcwd())


In [2]:
from src.data_loader import load_raw_data, profile_dataset
df = load_raw_data()
print(profile_dataset(df))
df.head()

Rows=5,110 | Cols=12 | Stroke=1: 249 (4.87%) | Stroke=0: 4,861 | BMI missing=201 | smoking_status='Unknown'=1,544 | Duplicate IDs=0


,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,9046,Male,67.0,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1
1,51676,Female,61.0,0,0,Yes,Self-employed,Rural,202.21,NaN,never smoked,1
2,31112,Male,80.0,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1
3,60182,Female,49.0,0,0,Yes,Private,Urban,171.23,34.4,smokes,1
4,1665,Female,79.0,1,0,Yes,Self-employed,Rural,174.12,24.0,never smoked,1


## Class imbalance & missing data

In [3]:
from src import eda
display(eda.missing_value_report(df))
display(eda.class_balance_report(df))

,feature,dtype,n_missing,pct_missing,n_unknown_token,pct_unknown_token,n_unique
0,bmi,float64,201,0.039335,0,0.000000,418
1,id,int64,0,0.000000,0,0.000000,5110
2,age,float64,0,0.000000,0,0.000000,104
3,gender,object,0,0.000000,0,0.000000,3
4,hypertension,Int64,0,0.000000,0,0.000000,2
5,heart_disease,Int64,0,0.000000,0,0.000000,2
6,work_type,object,0,0.000000,0,0.000000,5
7,ever_married,object,0,0.000000,0,0.000000,2
8,Residence_type,object,0,0.000000,0,0.000000,2
9,avg_glucose_level,float64,0,0.000000,0,0.000000,3979


,level,n_total,n_positive,n_negative,positive_rate,imbalance_ratio
0,sample-level,5110,249,4861,0.048728,19.522088
1,patient-level,5110,249,4861,0.048728,19.522088


## Risk stratification by age and glucose level

In [4]:
display(eda.stroke_rate_by_age_band(df))
display(eda.glucose_risk_strata(df))
display(eda.correlation_with_target(df))

,age_band,n_samples,n_stroke,stroke_rate
0,"[0, 20)",966,2,0.00207
1,"[20, 40)",1204,6,0.004983
2,"[40, 50)",730,12,0.016438
3,"[50, 60)",834,48,0.057554
4,"[60, 70)",621,47,0.075684
5,"[70, 80)",569,94,0.165202
6,"[80, 120)",186,40,0.215054


,glucose_stratum,n_samples,n_stroke,stroke_rate,risk_ratio_vs_normal
0,<100 (normal),3131,112,0.035771,1.0
1,100-125 (prediabetes),998,37,0.037074,1.036421
2,126-199 (diabetes),547,44,0.080439,2.248694
3,>=200 (severe),434,56,0.129032,3.607143


,feature,corr_with_stroke
0,age,0.245257
1,heart_disease,0.134914
2,avg_glucose_level,0.131945
3,hypertension,0.127904
4,bmi,0.042374


## Demographic bias

In [5]:
display(eda.demographic_bias_report(df))

,attribute,category,n_samples,n_stroke,stroke_rate,share_of_cohort
0,gender,Male,2115,108,0.051064,0.413894
1,gender,Female,2994,141,0.047094,0.58591
2,gender,Other,1,0,0.0,0.000196
3,Residence_type,Urban,2596,135,0.052003,0.508023
4,Residence_type,Rural,2514,114,0.045346,0.491977
5,work_type,Self-employed,819,65,0.079365,0.160274
6,work_type,Private,2925,149,0.05094,0.572407
7,work_type,Govt_job,657,33,0.050228,0.128571
8,work_type,children,687,2,0.002911,0.134442
9,work_type,Never_worked,22,0,0.0,0.004305


## Overview figure

In [6]:
fig = eda.plot_eda_overview(df, save_path='reports/figures/eda_overview.png')

## Export all EDA tables to CSV (reports/eda/)

In [7]:
tables = eda.run_full_eda(df, output_dir='reports/eda')
list(tables)

['missing_values',
 'class_balance',
 'numeric_summary',
 'age_bands',
 'glucose_strata',
 'correlation',
 'demographic_bias',
 'comorbidity']